# Out-of-scope detection — eval notebook

Tests the two-pass out-of-scope detection pipeline against the `/nli` endpoint.

**Decision logic** (implemented here, not in the service):

1. **Whitelist pass (OR)** — call `/nli` with `[relevance_hypothesis]`, no threshold.  
   If score < `relevance_threshold` → `out_scope` (reason: `failed_whitelist`).
2. **Blacklist pass (AND-exclusion)** — call `/nli` with `blacklist_labels`, `threshold=blacklist_threshold, blacklist=True`.  
   If any score ≥ threshold → `out_scope` (reason: the triggering hypothesis).  
   The service short-circuits on the first trigger, so we may see fewer scores than labels.
3. Both passes clear → `in_scope`.

Test cases live in `eval/fixtures/nli-news-oos.json`.

In [9]:
import os, json, requests, yaml
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/nli'
HEADERS      = {'Content-Type': 'application/json', 'X-API-Key': f'{os.environ.get("NLP_API_KEY")}'}
TIMEOUT      = 120

cfg = yaml.safe_load(Path('../config/topics.yaml').read_text(encoding='utf-8'))
RELEVANCE_HYPOTHESIS  = cfg['relevance_hypothesis'].strip()
RELEVANCE_THRESHOLD   = cfg['relevance_threshold']
BLACKLIST_LABELS      = cfg['blacklist_labels']
BLACKLIST_THRESHOLD   = cfg['blacklist_threshold']

CASES = json.loads(Path('fixtures/nli-news-oos.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')
print(f'relevance_threshold={RELEVANCE_THRESHOLD}  blacklist_threshold={BLACKLIST_THRESHOLD}')
print(f'{len(BLACKLIST_LABELS)} blacklist labels')

8 cases -> https://wiig.dia.fi.upm.es/b4c_nlp/nli
relevance_threshold=0.4  blacklist_threshold=0.7
3 blacklist labels


In [10]:
def build_premise(inp):
    headline = (inp.get('headline') or '').strip()
    text     = (inp.get('text') or '').strip()
    return f'{headline}. {text}' if headline else text


def call_nli(premise, hypotheses, threshold=None, blacklist=False):
    body = {'text': premise, 'hypotheses': hypotheses}
    if threshold is not None:
        body['threshold'] = threshold
    if blacklist:
        body['blacklist'] = True
    r = requests.post(ENDPOINT, json=body, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json().get('scores', [])


def classify_oos(inp):
    """Return (verdict, trigger, whitelist_score, blacklist_scores)."""
    premise = build_premise(inp)

    # Pass 1 — whitelist
    wl_scores = call_nli(premise, [RELEVANCE_HYPOTHESIS])
    wl_score  = wl_scores[0]['score'] if wl_scores else 0.0
    if wl_score < RELEVANCE_THRESHOLD:
        return 'out_scope', 'failed_whitelist', wl_score, []

    # Pass 2 — blacklist (short-circuits on first trigger)
    bl_scores = call_nli(premise, BLACKLIST_LABELS,
                         threshold=BLACKLIST_THRESHOLD, blacklist=True)
    for item in bl_scores:
        if item['score'] >= BLACKLIST_THRESHOLD:
            return 'out_scope', 'blacklist', wl_score, bl_scores

    return 'in_scope', None, wl_score, bl_scores

In [11]:
results = []
for case in CASES:
    print(f"── {case['id']}: {case['description']}")
    try:
        verdict, trigger, wl_score, bl_scores = classify_oos(case['input'])
    except Exception as exc:
        print('   ERROR:', exc)
        print()
        results.append((case['id'], False))
        continue

    # Whitelist
    wl_pass = wl_score >= RELEVANCE_THRESHOLD
    wl_icon = '✅' if wl_pass else '❌'
    print(f'  Whitelist  score={wl_score:.3f}  threshold={RELEVANCE_THRESHOLD}  {wl_icon}')

    # Blacklist
    if bl_scores:
        print(f'  Blacklist  ({len(bl_scores)} evaluated, short-circuit at first trigger)')
        for item in bl_scores:
            flag = ' <-- TRIGGERED' if item['score'] >= BLACKLIST_THRESHOLD else ''
            print(f'    score={item["score"]:.3f}  "{item["hypothesis"]}"{flag}')
    elif wl_pass:
        print('  Blacklist  (not evaluated — whitelist failed first)'  
              if trigger == 'failed_whitelist' else
              '  Blacklist  no trigger found')

    print(f'  Verdict: {verdict}' + (f'  trigger={trigger!r}' if trigger else ''))

    # Score
    exp = case['expected']
    verdict_ok  = verdict == exp['verdict']
    trigger_ok  = trigger == exp['trigger']
    passed      = verdict_ok and trigger_ok
    icon        = '✅' if passed else '❌'
    mismatch    = []
    if not verdict_ok:
        mismatch.append(f'expected verdict={exp["verdict"]!r}')
    if not trigger_ok:
        mismatch.append(f'expected trigger={exp["trigger"]!r}')
    note = '  ' + ', '.join(mismatch) if mismatch else ''
    print(f'  {icon} case {case["id"]}{note}')
    print()
    results.append((case['id'], passed))

── oos-001: Bike lane opening in Seville — in_scope
  Whitelist  score=0.710  threshold=0.4  ✅
  Blacklist  (3 evaluated, short-circuit at first trigger)
    score=0.021  "Este artículo trata sobre ciclismo deportivo, competición ciclista o carreras de bicicletas."
    score=0.001  "Este artículo trata sobre senderismo, montañismo o deporte en la naturaleza sin relación con la movilidad urbana."
    score=0.001  "Este artículo trata sobre un evento deportivo o competición atlética, sin relación con infraestructura urbana."
  Verdict: in_scope
  ✅ case oos-001

── oos-002: Tour de España race coverage — out_scope via blacklist (ciclismo deportivo)
  Whitelist  score=0.006  threshold=0.4  ❌
  Verdict: out_scope  trigger='failed_whitelist'
  ❌ case oos-002  expected trigger='blacklist'

── oos-003: Mountain biking trail article — out_scope via blacklist (senderismo/montaña)
  Whitelist  score=0.008  threshold=0.4  ❌
  Verdict: out_scope  trigger='failed_whitelist'
  ❌ case oos-003  expect

In [12]:
passed_n = sum(1 for _, p in results if p)
print('SCORECARD · /nli  (out-of-scope detection)')
print(f'  Cases passing: {passed_n}/{len(results)}')
for cid, p in results:
    print(f"   {'✅' if p else '❌'} {cid}")

SCORECARD · /nli  (out-of-scope detection)
  Cases passing: 4/8
   ✅ oos-001
   ❌ oos-002
   ❌ oos-003
   ✅ oos-004
   ❌ oos-005
   ✅ oos-006
   ❌ oos-007
   ✅ oos-008
